# 06 · Freeze the models (Levels 1–3) before any test is scored

Sandesh Lamichhane, Assistant Professor, Yenepoya (Deemed to be University).

**Run this after notebooks 04 and 05 have finished.** The sealed-test design (ADR-0007) depends on this step. It:

1. records your **prior-exposure declaration** (step 2). The freeze refuses to run without it, so the declaration
   is fixed before any test number exists;
2. applies the preregistered rule to the **validation** results, once, and writes `results/models/freeze.json`
   (Levels 1–2) and `results/models/freeze_lanl.json` (Level 3), with the SHA-256 of every model file and
   development record;
3. saves both to GitHub and checks that the test code will accept them.

No model is trained, and no test or target data is read.

**Expected result** (from the validation records already on GitHub):

* **Levels 1–2: `random_forest`.** XGBoost has the highest val macro-F1 (0.999754). Random forest (0.999620) and
  the MLP (0.999421) are within the preregistered margin of 0.0005, so the earliest of the three in config order
  wins. For H1c it is random forest in the `unsw_canonical` space.
* **Level 3: `logreg`**, with a val early-alert rate of 0.349206 (22 of 63 hosts at 1 % false alarms). It is
  frozen as preregistered, as you decided on 2026-10-26.

If the notebook prints anything else, stop and send the output to the project thread.

**Before you start:** a standard CPU runtime is enough (*Runtime → Change runtime type → CPU*); no GPU is needed.

**The freeze runs once.** Steps 4 and 5 never overwrite a freeze. If a step has already run, it says so and
moves on, so running the cells again is safe. After a disconnect, run step 1 again, then continue from the
step that was interrupted.

| Step | What it does | Approximate time |
|---|---|---|
| 1 | connect Drive and GitHub, update the code, run the tests (expected `in sync with GitHub at …` and `… passed` with no failures) | 3 min |
| 2 | **your prior-exposure declaration**: type three answers in the boxes, then run | 5–10 min |
| 3 | development tables at six decimals (expected `all 14 candidate records complete with 5 seeds`) | 1 min |
| 4 | freeze Levels 1–2 (hashes 50 model files on Drive) | 5–30 min |
| 5 | freeze Level 3 (hashes 20 model files) | 2–10 min |
| 6 | check that the freeze records are on GitHub unchanged and every model file still matches | 5–30 min |

**Where the results go.** Everything is saved in the Drive copy of the repository
(`MyDrive/PhD-AI-Cyber/code/phd-ai-threat-detection/`) and on GitHub. There is nothing to move by hand.

* `docs/protocol/prior_exposure.md`: your declaration
* `results/models/freeze.json` and `results/models/freeze_lanl.json`: the freeze records
* `results/models/freeze_summary.md`: both freezes in words, to six decimals (for the methods chapter)
* `results/models/development/summary.md` and `lanl_summary.md`: the development tables, to six decimals
* `results/models/logs/<time>_freeze.log` and `<time>_freeze_lanl.log`: run logs

The model files stay in `MyDrive/PhD-AI-Cyber/models/development/` (Drive only); their SHA-256 values are in the
freeze records on GitHub. **From now on, do not open, move or re-save any model file.** The test code checks
every one against its SHA-256 and refuses to score if one has changed.

**When it finishes:** use *File → Download → Download .ipynb* and send the finished notebook to the project
thread. The next notebook scores the sealed test.

## Step 1 · Connect and update (expected end: `in sync with GitHub at …` and `… passed` with no failures)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import datetime as _dt, json, os, pathlib, shutil, subprocess, time

DRIVE_ROOT = '/content/drive/MyDrive/PhD-AI-Cyber'
REPO_DIR   = f'{DRIVE_ROOT}/code/phd-ai-threat-detection'
os.environ['PHD_DRIVE_ROOT'] = DRIVE_ROOT
os.environ['PHD_SCRATCH'] = '/content/scratch'
TOKEN  = userdata.get('GITHUB_TOKEN')
REMOTE = f'https://{TOKEN}@github.com/sandesh20lamichhane/phd-ai-threat-detection.git'

# Git keeps its database on the Colab disk, never on Drive: Drive's file system can crash git while it unpacks
# downloads ("unpack-objects died of signal 7"). The files themselves stay in the Drive folder, so every result
# is in Drive the moment it is written, and the sync below copies it to GitHub.
GIT_DIR = '/content/phd-repo.git'
STATE = pathlib.Path(DRIVE_ROOT, 'code', '.phd_sync_state.json')    # commit the Drive folder last matched
os.environ['GIT_DIR'], os.environ['GIT_WORK_TREE'] = GIT_DIR, REPO_DIR
RECORD_DIRS = ('results/', 'docs/data_cards/', 'config/feature_mapping/')
_trust_drive_records = True

def _mask(text):
    return text.replace(TOKEN or '\0', '***').strip()

def git(*args, check=False):
    r = subprocess.run(['git', *args], capture_output=True, text=True, cwd=REPO_DIR)
    out = _mask(r.stdout + r.stderr)
    if check and r.returncode != 0:
        print(out)
        raise SystemExit(f'git {args[0]} failed (see message above)')
    return r.returncode, out

def _stamp():
    return _dt.datetime.now(_dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ')

def _last_synced_commit():
    """The commit the Drive folder was last synced to: from the state file, else from the old .git on Drive."""
    try:
        if STATE.exists():
            return json.loads(STATE.read_text()).get('commit')
        g = pathlib.Path(REPO_DIR, '.git')
        head = (g / 'HEAD').read_text().strip()
        if not head.startswith('ref: '):
            return head
        ref = head[5:]
        if (g / ref).exists():
            return (g / ref).read_text().strip()
        for line in (g / 'packed-refs').read_text().splitlines():
            if line.endswith(' ' + ref):
                return line.split()[0]
    except (OSError, ValueError):
        pass
    return None

def setup_git():
    """Once per Colab session: download the git database to the Colab disk and line it up with the Drive folder."""
    global _trust_drive_records
    if pathlib.Path(GIT_DIR, 'HEAD').exists():
        return
    pathlib.Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
    env = {k: v for k, v in os.environ.items() if k not in ('GIT_DIR', 'GIT_WORK_TREE')}
    r = subprocess.run(['git', 'clone', '--bare', '--quiet', REMOTE, GIT_DIR], capture_output=True, text=True, env=env)
    if r.returncode != 0:
        print(_mask(r.stdout + r.stderr))
        raise SystemExit('git clone failed (see message above)')
    git('config', 'core.bare', 'false', check=True)
    git('config', 'core.filemode', 'false')
    old = _last_synced_commit()
    known = bool(old) and git('cat-file', '-e', f'{old}^{{commit}}')[0] == 0
    git('reset', '-q', old if known else 'HEAD', check=True)      # sets the index only; Drive files untouched
    _trust_drive_records = known
    if not known:
        print('note: could not tell which commit the Drive folder last matched; GitHub wins for every tracked '
              'file, and Drive versions that differ are backed up first')
    dot = pathlib.Path(REPO_DIR, '.git')
    if dot.exists():
        dest = pathlib.Path(DRIVE_ROOT, 'backups', f'old_drive_git_{_stamp()}')
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(dot), str(dest))
        print(f'old git folder on Drive moved to {dest} (no longer used)')
    print('git database on the Colab disk; Drive folder last matched ' + (old[:7] if known else 'an unknown commit'))

def _push_hint(out):
    o = out.lower()
    if any(k in o for k in ('authentication failed', 'invalid username', 'invalid credentials', '403', 'permission')):
        return ('GitHub did not accept the token. If it has expired, create a new fine-grained token (repository '
                'phd-ai-threat-detection, Contents: Read and write) and save it as the Colab secret GITHUB_TOKEN.')
    if 'rejected' in o or 'fetch first' in o or 'non-fast-forward' in o:
        return 'GitHub had a newer commit; fetching it and trying again.'
    return 'GitHub or the network did not respond; trying again in a moment.'

def sync(message=None, strict=True, also=()):
    """Make the Drive copy of the repo identical to GitHub, keeping every result produced here.

    Never merges or rebases, so open notebooks or other local edits cannot block it:
    1. remember result files changed here (results/, docs/data_cards/, config/feature_mapping/)
    2. back up every other local change to MyDrive/PhD-AI-Cyber/backups/sync_<time>/
    3. reset the copy to GitHub's main, put the result files back, commit and push them
    Results are on Drive before this runs. If GitHub cannot be reached, strict=False only warns: the next
    sync (the next step, or step 1 in a new session) saves the same files.
    `also` names other files changed here on purpose (e.g. docs/protocol/prior_exposure.md), saved like results.
    """
    global _trust_drive_records
    setup_git()
    git('config', 'user.name', 'Sandesh Lamichhane')
    git('config', 'user.email', 'san30lamichhane@gmail.com')
    tries, out = 5, ''
    for attempt in range(1, tries + 1):
        code, out = git('fetch', '--quiet', REMOTE, 'main')
        if code == 0:
            base = git('merge-base', 'HEAD', 'FETCH_HEAD')[1] or 'FETCH_HEAD'
            changed = set(git('diff', '--name-only', base)[1].splitlines())
            untracked = set(git('ls-files', '--others', '--exclude-standard')[1].splitlines())
            # untracked non-record files survive the reset unless GitHub has a file at the same path
            untracked = {f for f in untracked if f.startswith(RECORD_DIRS) or git('cat-file', '-e', f'FETCH_HEAD:{f}')[0] == 0}
            changed |= untracked
            is_file = lambda f: pathlib.Path(REPO_DIR, f).is_file()
            keep = sorted(f for f in changed if is_file(f) and (f in also or f.startswith(RECORD_DIRS)
                          and (_trust_drive_records or f in untracked)))
            other = sorted(f for f in changed if f not in keep and is_file(f))
            stamp = _stamp()
            hold = pathlib.Path('/content/sync_hold') / stamp
            for f in keep:
                (hold / f).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(pathlib.Path(REPO_DIR, f), hold / f)
            if other:
                bak = pathlib.Path(DRIVE_ROOT, 'backups', f'sync_{stamp}')
                for f in other:
                    (bak / f).parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(pathlib.Path(REPO_DIR, f), bak / f)
                print(f'{len(other)} other local change(s) backed up to {bak} (e.g. notebook outputs)')
            git('reset', '--hard', '--quiet', 'FETCH_HEAD', check=True)
            for f in keep:
                dest = pathlib.Path(REPO_DIR, f)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(hold / f, dest)
            if keep:
                git('add', '--', *keep, check=True)
                if git('diff', '--cached', '--quiet')[0] != 0:
                    git('commit', '-q', '-m', message or 'Save results produced in Colab', check=True)
            code, out = git('push', '--quiet', REMOTE, 'HEAD:main')
            if code == 0:
                head = git('log', '--oneline', '-1')[1]
                STATE.write_text(json.dumps({'commit': git('rev-parse', 'HEAD')[1], 'utc': stamp}))
                _trust_drive_records = True
                print(f'in sync with GitHub at {head}' + (f'; {len(keep)} result file(s) saved' if keep else ''))
                return True
        print(f'GitHub refused the save (try {attempt} of {tries}): {out or "no message"}')
        print('  ' + _push_hint(out))
        if attempt < tries:
            time.sleep(min(60, 4 * 2 ** (attempt - 1)))
    msg = ('Could not save to GitHub. Nothing is lost: every result is in the Drive folder, and the next save '
           '(the next step, or step 1 after reopening the notebook) sends it to GitHub. Copy the messages above '
           'into the project thread if this keeps happening.')
    if strict:
        raise SystemExit(msg)
    print('WARNING: ' + msg)
    return False


sync('Save results from an earlier Colab run')      # update the code; results made here are kept
assert pathlib.Path(REPO_DIR, 'src/models/seal.py').exists()
%cd $REPO_DIR
!pip -q install -r requirements.txt
!python -m pytest -q tests
import sys
sys.path.insert(0, REPO_DIR)

def push(message, also=()):
    """Save results from the Drive copy of the repo to GitHub, so both hold the same records.
    If GitHub cannot be reached it only warns: the results are in Drive and the next save sends them."""
    if sync(message, strict=False, also=also):
        print('saved to GitHub:', message)

def develop(*args, message):
    """Run scripts/develop.py (its output shows below); push the records only if it succeeded."""
    ip = get_ipython()
    ip.system('python scripts/develop.py ' + ' '.join(args))
    ok = ip.user_ns.get('_exit_code', 1) == 0
    push(message if ok else message + ' (FAILED, log only)')
    if not ok:
        raise SystemExit('This step failed. Copy the error above into the project thread.')

## Step 2 · Your prior-exposure declaration (type in the three boxes, then run the cell)

Reviewers must be able to tell which results are prospective. Before 25 Oct 2026 you used CICIDS2017,
CSE-CIC-IDS2018 and UNSW-NB15 in earlier projects (`ids-adapt`, `when-should-ids-adapt`,
`cicids-label-correction-audit`). `docs/protocol/prior_exposure.md` has three items that only you can fill in.
Type one answer per box, in full sentences:

* **CICIDS2017**: which splits you used (random, by day, other), which models you trained, and the headline
  scores you saw.
* **CSE_CIC_IDS2018**: whether you ever trained on CICIDS2017 and tested on CSE-CIC-IDS2018 (or the reverse), and
  if so, how large the drop was. Also say whether your `ids-adapt` copy was the original or the corrected release.
  If you never did, write that no such transfer result was computed or seen.
* **UNSW_NB15**: the same, for CICIDS2017 → UNSW-NB15.

If you do not remember a number exactly, say so and give your best recollection (for example *"about 0.3, not
remembered exactly"*). An honest approximate answer is what reviewers need; an empty box is refused. The answers
go into the file word for word, dated, and are saved to GitHub before the freeze. The freeze then records the
file's SHA-256.

In [ ]:
#@title Step 2 · Prior exposure: type your answers in the boxes on the right, then run this cell
CICIDS2017 = ""  #@param {type:"string"}
CSE_CIC_IDS2018 = ""  #@param {type:"string"}
UNSW_NB15 = ""  #@param {type:"string"}

import re
from IPython.display import Markdown, display
PE_REL = 'docs/protocol/prior_exposure.md'
PE = pathlib.Path(REPO_DIR, PE_REL)
text = PE.read_text()
if 'Author to complete' not in text:
    print('The declaration is already complete; saving it to GitHub if it is not there yet.')
else:
    answers = {'CICIDS2017': CICIDS2017, 'CSE-CIC-IDS2018': CSE_CIC_IDS2018, 'UNSW-NB15': UNSW_NB15}
    empty = [k for k, v in answers.items() if len(v.strip()) < 10]
    if empty:
        raise SystemExit(f'Type your answer for {empty} in the boxes of this cell first (a full sentence each).')
    today = _dt.datetime.now(_dt.timezone.utc).strftime('%Y-%m-%d')
    lines = []
    for line in text.splitlines():
        for name, ans in answers.items():
            if line.startswith(f'| {name} |') and 'Author to complete' in line:
                decl = f"**Author's declaration ({today}):** " + ' '.join(ans.split()).replace('|', '/') + ' |'
                line = re.sub(r'\*\*Author to complete:\*\*.*\|\s*$', lambda m: decl, line)
        lines.append(line)
    text = '\n'.join(lines).rstrip() + (
        f'\n\n**Declaration.** The three items above were completed by Sandesh Lamichhane on {today} (UTC) in '
        'notebook 06, step 2, after model development on validation data and before the models were frozen; no '
        'test or target score existed. The SHA-256 of this file is recorded in `results/models/freeze.json` and '
        '`results/models/freeze_lanl.json`.\n')
    if 'Author to complete' in text:
        raise SystemExit('An item was not filled in; copy this message into the project thread.')
    PE.write_text(text)
display(Markdown(PE.read_text()))
sync('Complete the prior-exposure declaration (before the freeze)', strict=True, also=(PE_REL,))

## Step 3 · Development tables at six decimals (expected: `all 14 candidate records complete with 5 seeds`)

In [ ]:
develop('--status', message='Levels 1-2 development table at six decimals')
develop('--lanl', '--status', message='Level 3 development table at six decimals')
from src.models import develop as D, lanl_develop as L
bad = ([f"{r['feature_space']}/{r['candidate']}" for r in D.summary_table() if not r['complete'] or r['seeds_done'] != 5]
       + [f"lanl/{r['candidate']}" for r in L.summary_table() if not r['complete'] or r['seeds_done'] != 5])
if bad:
    raise SystemExit(f'development is not complete for {bad}: finish notebook 04 or 05 first')
print('all 14 candidate records complete with 5 seeds')
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/development/summary.md').read_text()))
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/development/lanl_summary.md').read_text()))

## Step 4 · Freeze Levels 1–2 (expected: `proposed detector: random_forest`)

Expected line: `proposed detector: random_forest (val macro-F1 0.999620; best 0.999754, candidates within 0.0005
of it: ['random_forest', 'xgboost', 'mlp'], the first in config order wins)`, then `in sync with GitHub at …`.

In [ ]:
FREEZE_L12 = pathlib.Path(REPO_DIR, 'results/models/freeze.json')
if FREEZE_L12.exists():
    print('Levels 1-2 are already frozen (the freeze runs once); showing the record.')
else:
    develop('--freeze', message='Freeze Levels 1-2: proposed detector chosen on validation by the preregistered rule (ADR-0007)')
sync('Save the Levels 1-2 freeze', strict=True)        # stops here if GitHub does not have the freeze yet
fr = json.loads(FREEZE_L12.read_text())
print('proposed detector:', fr['proposed'], '|', fr['proposed_by_hypothesis'])
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/freeze_summary.md').read_text()))

## Step 5 · Freeze Level 3 (expected: `Level 3 predictive model: logreg`)

Expected line: `Level 3 predictive model: logreg (val early-alert rate 0.349206, PR-AUC 0.033527)`, then
`in sync with GitHub at …`. This applies your decision of 2026-10-26: Level 3 is frozen as preregistered, with
no further development round.

In [ ]:
FREEZE_L3 = pathlib.Path(REPO_DIR, 'results/models/freeze_lanl.json')
if FREEZE_L3.exists():
    print('Level 3 is already frozen (the freeze runs once); showing the record.')
else:
    develop('--lanl', '--freeze', message='Freeze Level 3: predictive model chosen on LANL validation by the preregistered rule (ADR-0007)')
sync('Save the Level 3 freeze', strict=True)
fl = json.loads(FREEZE_L3.read_text())
print('Level 3 predictive model:', fl['proposed'])
display(Markdown(pathlib.Path(REPO_DIR, 'results/models/freeze_summary.md').read_text()))

## Step 6 · Check the freeze (expected: two `OK` lines)

This step runs the same check the test-scoring code will run: each freeze record must be committed to git
unchanged, and every model file and development record must still match its SHA-256. Expected:
`OK Levels 1-2: random_forest … 50 model files and 10 development records match` and
`OK Level 3: logreg … 20 model files and 4 development records match`.

In [ ]:
import importlib
from src.models import seal
importlib.reload(seal)
for track, label in (('l12', 'Levels 1-2'), ('lanl', 'Level 3')):
    f = seal.require_frozen(track=track)
    print(f"OK {label}: {f['proposed']}, frozen {f['frozen_utc']}, committed to git unchanged; "
          f"{len(f['model_files_sha256'])} model files and {len(f['development_records_sha256'])} development "
          f"records match their SHA-256; prior-exposure declaration {f['prior_exposure_sha256'][:12]}…")
print(git('log', '--oneline', '-4', '--', 'results/models/freeze.json', 'results/models/freeze_lanl.json',
          'docs/protocol/prior_exposure.md')[1])
print('On GitHub: https://github.com/sandesh20lamichhane/phd-ai-threat-detection/tree/main/results/models')
print('Done. Download this notebook (File -> Download -> Download .ipynb) and send it to the project thread.')